In [2]:
import pandas as pd
import numpy as np

# keep_default_na=False : sinon pandas transforme le texte "None" en valeur manquante
raw = pd.read_csv("diabetic_data.csv", keep_default_na=False, low_memory=False)

## 1. Business Understanding

**Problématique :** à partir des informations disponibles à la sortie, peut-on repérer les patients diabétiques à risque de réadmission à 30 jours, comprendre quels profils existent, et orienter le suivi vers les bons patients ? Dans nos données, **11,2 %** des séjours sont suivis d'une réadmission à 30 jours.

### 1.1 État de l'art

**Ce qui existe déjà**
- **Enjeu :** les réadmissions précoces sont coûteuses et nuisibles au patient. Aux États-Unis, des programmes de pénalités (type **HRRP**) sanctionnent les hôpitaux dont les taux sont trop élevés.
- **Ce jeu de données :** *Diabetes 130-US Hospitals (1999-2008)* provient de l'étude de Strack et al. (2014). Elle examine l'effet de la mesure de l'HbA1c sur la réadmission, par régression logistique. Elle conclut que cet effet dépend du diagnostic principal.
- **Modèles prédictifs :** la littérature sur la prédiction des réadmissions rapporte en général un pouvoir prédictif modeste (AUC ≈ 0,6 à 0,7). Les variables les plus citées sont l'historique d'hospitalisations, la durée de séjour et les comorbidités.
- **Équité :** un bon score global peut cacher des performances plus faibles selon l'âge, le genre ou l'origine des patients.

**Où se situe notre travail**
- Nous reprenons la question de la réadmission, mais en la traitant de bout en bout : **profils** de patients, **score de risque**, **facteurs explicatifs**, **priorisation** selon la capacité de l'équipe, **tableau de bord** et **audit d'équité**.
- Un AUC proche de 0,95 serait un signal de fuite de données, pas une réussite.

*Références : Strack B. et al., « Impact of HbA1c measurement on hospital readmission rates », BioMed Research International, 2014, doi:10.1155/2014/781670.*

### 1.2 Objectifs métier (BO)

| BO | Objectif métier | Idée |
|---|---|---|
| **BO1** | Identifier les différents profils de patients | décrire des profils types avec un nom parlant (« séjour express », « revenant »…) |
| **BO2** | Étudier la réadmission hospitalière | donner à chaque patient un score de risque avant la sortie (feu tricolore) |
| **BO3** | Identifier les caractéristiques associées à la réadmission | comprendre quels patients et quels séjours reviennent, pour en informer les médecins |
| **BO4** | Identifier les patients nécessitant une attention particulière | concentrer les appels de suivi là où l'impact est maximal |
| **BO5** | Tableau de bord qualité et suivi du diabète | fournir à la direction des indicateurs par profil, âge, destination de sortie et prise en charge du diabète |
| **BO6** | IA de confiance | s'assurer que le modèle est explicable et équitable |

### 1.3 Objectifs Data Science (DSO)

| BO | Objectif Data Science (DSO) |
|---|---|
| **BO1** | regrouper les patients en profils homogènes et interprétables, puis mesurer le taux de réadmission de chaque profil |
| **BO2** | prédire `readmitted_30` à partir des informations connues à la sortie |
| **BO3** | mesurer l'influence de chaque variable sur le risque et la traduire en messages compréhensibles par les médecins |
| **BO4** | classer les patients par score, choisir le nombre de patients suivis selon la capacité de l'équipe et mesurer le gain par rapport à un ciblage au hasard |
| **BO5** | calculer des indicateurs agrégés et tester si l'HbA1c, les changements de traitement et l'insuline sont liés à la réadmission |
| **BO6** | comparer les performances du modèle selon le genre, la race et la tranche d'âge |

**Cible :** `readmitted_30 = 1` si `readmitted == "<30"`, sinon 0 (définition des programmes de type HRRP). Limite : on perd la distinction « réadmis après 30 jours » (31 % des séjours).

**Critères de succès :** *utile* (le score capte nettement plus de réadmissions qu'un ciblage au hasard), *compréhensible* (profils et facteurs en langage médical), *juste* (pas d'écart marqué entre sous-groupes), *réaliste* (AUC ≈ 0,6 à 0,7).